# Multi-Angle Face Verification Testing

**Goal**: Evaluate face verification performance across multiple face angles using RetinaFace for detection and ArcFace for embeddings.

**Angles tested**
- front face
- left 45 deg
- right 45 deg
- upward angle
- downward angle

**What this notebook delivers**
- Embeddings per angle
- Similarity analysis for genuine vs imposter comparisons
- Tables, heatmaps, and charts
- Angle difficulty analysis and recommendations

In [ ]:
import os
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load Multi-Angle Dataset and Prepare Embeddings

This section loads images arranged by identity and angle, then generates embeddings using RetinaFace + ArcFace.

**Expected folder layout**
```
data/multi_angle/
  person_01/
    front.jpg
    left_45.jpg
    right_45.jpg
    up.jpg
    down.jpg
  person_02/
    ...
```

If these files are not present or the models are not installed, the notebook falls back to a simulated dataset to keep the analysis reproducible.

In [ ]:
# Optional dependencies for real data processing
try:
    import cv2
except ImportError:
    cv2 = None

try:
    from retinaface import RetinaFace
except ImportError:
    RetinaFace = None

try:
    import insightface
except ImportError:
    insightface = None

ANGLES = ["front", "left_45", "right_45", "up", "down"]
DATA_DIR = Path("data/multi_angle")


def cosine_similarity(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def detect_first_face(img_bgr):
    if RetinaFace is None or img_bgr is None:
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = RetinaFace.detect_faces(img_rgb)
    if not isinstance(results, dict):
        return None
    for _, info in results.items():
        x1, y1, x2, y2 = info.get("facial_area", [0, 0, 0, 0])
        return img_bgr[max(y1, 0):max(y2, 0), max(x1, 0):max(x2, 0)]
    return None


_arcface_model = None
def arcface_embedding(face_bgr):
    global _arcface_model
    if insightface is None or face_bgr is None:
        return None
    if _arcface_model is None:
        _arcface_model = insightface.model_zoo.get_model("arcface_r100_v1")
        _arcface_model.prepare(ctx_id=-1)
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    face_rgb = cv2.resize(face_rgb, (112, 112))
    emb = _arcface_model.get_feat(face_rgb).flatten()
    emb = emb / np.linalg.norm(emb)
    return emb


def load_real_embeddings():
    if cv2 is None or RetinaFace is None or insightface is None:
        return None
    if not DATA_DIR.exists():
        return None
    embeddings = {}
    for person_dir in sorted(DATA_DIR.iterdir()):
        if not person_dir.is_dir():
            continue
        identity = person_dir.name
        embeddings[identity] = {}
        for angle in ANGLES:
            img_path = person_dir / f"{angle}.jpg"
            if not img_path.exists():
                continue
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                continue
            face = detect_first_face(img_bgr)
            emb = arcface_embedding(face) if face is not None else None
            if emb is not None:
                embeddings[identity][angle] = emb
        if len(embeddings[identity]) == 0:
            embeddings.pop(identity)
    return embeddings if embeddings else None


def simulate_embeddings(num_identities=30, dim=512):
    # Angle-specific noise to simulate pose difficulty
    angle_noise = {
        "front": 0.10,
        "left_45": 0.16,
        "right_45": 0.16,
        "up": 0.20,
        "down": 0.24,
    }
    embeddings = {}
    base = np.random.randn(num_identities, dim)
    base = base / np.linalg.norm(base, axis=1, keepdims=True)
    for i in range(num_identities):
        identity = f"person_{i:02d}"
        embeddings[identity] = {}
        for angle in ANGLES:
            vec = base[i] + angle_noise[angle] * np.random.randn(dim)
            vec = vec / np.linalg.norm(vec)
            embeddings[identity][angle] = vec
    return embeddings


embeddings_by_id = load_real_embeddings()
source = "real" if embeddings_by_id is not None else "simulated"
if embeddings_by_id is None:
    embeddings_by_id = simulate_embeddings()

print(f"Embeddings source: {source}")
print("Identities:", len(embeddings_by_id))

availability = pd.DataFrame(
    {identity: {angle: (angle in data) for angle in ANGLES}
     for identity, data in embeddings_by_id.items()}
).T
availability.head()

## 2. Compute Similarity Scores for Genuine vs Imposter Comparisons

We compare:
- **Same person, different angles** (genuine cross-angle matches)
- **Different person, same angle** (imposter same-angle matches)

Cosine similarity is used to quantify embedding similarity. Higher scores imply closer identity matches.

In [ ]:
identities = list(embeddings_by_id.keys())

# Genuine: same person, different angles
genuine_scores = {angle_a: {angle_b: [] for angle_b in ANGLES} for angle_a in ANGLES}
for identity in identities:
    data = embeddings_by_id[identity]
    for angle_a, angle_b in combinations(ANGLES, 2):
        if angle_a in data and angle_b in data:
            score = cosine_similarity(data[angle_a], data[angle_b])
            genuine_scores[angle_a][angle_b].append(score)
            genuine_scores[angle_b][angle_a].append(score)

# Imposter: different person, same angle
imposter_scores = {angle: [] for angle in ANGLES}
for angle in ANGLES:
    available = [i for i in identities if angle in embeddings_by_id[i]]
    for i, j in combinations(available, 2):
        score = cosine_similarity(embeddings_by_id[i][angle], embeddings_by_id[j][angle])
        imposter_scores[angle].append(score)

# Aggregate mean similarity tables
genuine_mean = pd.DataFrame(
    {a: {b: (np.mean(genuine_scores[a][b]) if genuine_scores[a][b] else np.nan)
         for b in ANGLES}
     for a in ANGLES}
)
imposter_mean = pd.Series({
    angle: (np.mean(scores) if scores else np.nan)
    for angle, scores in imposter_scores.items()
}, name="Imposter Mean Similarity")

genuine_mean, imposter_mean

## 3. Similarity Score Tables and Heatmaps

The tables summarize average similarity scores. Heatmaps visualize how angles relate to each other for the same identity.

In [ ]:
print("Genuine (same person, different angles) mean similarity")
display(genuine_mean)

print("Imposter (different person, same angle) mean similarity")
display(imposter_mean.to_frame())

# Heatmap for genuine cross-angle similarity
plt.figure(figsize=(6, 5))
plt.imshow(genuine_mean.values, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Cosine similarity")
plt.xticks(range(len(ANGLES)), ANGLES, rotation=30)
plt.yticks(range(len(ANGLES)), ANGLES)
plt.title("Genuine Cross-Angle Similarity (Mean)")
plt.tight_layout()
plt.show()

# Bar chart for imposter same-angle similarity
plt.figure(figsize=(6, 4))
plt.bar(imposter_mean.index, imposter_mean.values, color="#8da0cb")
plt.title("Imposter Same-Angle Similarity (Mean)")
plt.ylabel("Cosine similarity")
plt.xticks(rotation=30)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 4. Angle-Wise Verification Accuracy

We estimate accuracy for each angle using a fixed threshold. Genuine pairs include that angle vs other angles. Imposter pairs are different identities with the same angle.

In [ ]:
THRESHOLD = 0.5

def compute_metrics_for_angle(angle, threshold):
    # Genuine: identity same, angle vs other angles
    genuine = []
    for identity in identities:
        data = embeddings_by_id[identity]
        if angle not in data:
            continue
        for other_angle in ANGLES:
            if other_angle == angle or other_angle not in data:
                continue
            genuine.append(cosine_similarity(data[angle], data[other_angle]))

    # Imposter: different identities, same angle
    imposter = []
    available = [i for i in identities if angle in embeddings_by_id[i]]
    for i, j in combinations(available, 2):
        imposter.append(cosine_similarity(embeddings_by_id[i][angle], embeddings_by_id[j][angle]))

    if len(genuine) == 0 or len(imposter) == 0:
        return np.nan, np.nan, np.nan, len(genuine), len(imposter)

    genuine = np.array(genuine)
    imposter = np.array(imposter)

    tp = np.sum(genuine >= threshold)
    fn = np.sum(genuine < threshold)
    fp = np.sum(imposter >= threshold)
    tn = np.sum(imposter < threshold)

    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    return acc, far, frr, len(genuine), len(imposter)


rows = []
for angle in ANGLES:
    acc, far, frr, g_n, i_n = compute_metrics_for_angle(angle, THRESHOLD)
    rows.append({
        "Angle": angle,
        "Accuracy": acc,
        "FAR": far,
        "FRR": frr,
        "Genuine Pairs": g_n,
        "Imposter Pairs": i_n,
    })

angle_metrics = pd.DataFrame(rows)
angle_metrics

In [ ]:
# Charts for accuracy and error rates by angle
plt.figure(figsize=(6, 4))
plt.bar(angle_metrics["Angle"], angle_metrics["Accuracy"], color="#66c2a5")
plt.title("Accuracy by Angle")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(angle_metrics["Angle"], angle_metrics["FAR"], marker="o", label="FAR")
plt.plot(angle_metrics["Angle"], angle_metrics["FRR"], marker="o", label="FRR")
plt.title("FAR/FRR by Angle")
plt.ylabel("Rate")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Angle Difficulty Analysis

We identify the angle with the highest accuracy (easiest) and the lowest accuracy (hardest) based on the computed metrics.

In [ ]:
valid_metrics = angle_metrics.dropna(subset=["Accuracy"])
best_row = valid_metrics.loc[valid_metrics["Accuracy"].idxmax()]
worst_row = valid_metrics.loc[valid_metrics["Accuracy"].idxmin()]

print("Highest accuracy angle:", best_row["Angle"], "->", float(best_row["Accuracy"]))
print("Hardest angle:", worst_row["Angle"], "->", float(worst_row["Accuracy"]))

## 6. Discussion: Multi-Angle vs Single Frontal Enrollment

**Why multi-angle enrollment improves robustness**
- It captures identity features across pose changes, making the embedding space more stable when the camera angle varies.
- It reduces the gap between probe images and enrollment references, lowering false rejects.

**Why single frontal enrollment causes false rejection**
- Non-frontal probes often introduce self-occlusion and pose distortion, reducing similarity to a purely frontal template.
- The cosine similarity drops for genuine pairs when the pose mismatch is large, pushing scores below the acceptance threshold.

## 7. Conclusion and Recommendations

- Use multi-angle enrollment when the deployment environment includes varied camera heights or user movement.
- Prioritize the angles that show the highest accuracy in your dataset for faster verification.
- If a single enrollment image is required, prefer a mild side angle (if it performs well) rather than strict frontal-only, and tune the threshold accordingly.

Include the angle metrics table and charts in your report to justify the final recommendation.